# Offshore Update — SL x City mapping (no pandas)

In [ ]:
import re
import glob
import os
from openpyxl import load_workbook

## Config — edit paths here if needed

In [ ]:
BASE_DIR = "/Users/ko20689900/Documents/Exersice"
BENCH_DATA_PATH = os.path.join(BASE_DIR, "BenchData.xlsx")
OFFSHORE_DIR = os.path.join(BASE_DIR, "offshore")
SHEET_NAME = "Sheet1"

COUNT_HEADER = "Count"
EMP_HEADER = "Employee Numbers"

## Core functions

In [ ]:
def load_bench_data(path):
    """Read BenchData.xlsx into a plain list of dicts using openpyxl only (no pandas)."""
    wb = load_workbook(path, data_only=True)
    ws = wb[SHEET_NAME] if SHEET_NAME in wb.sheetnames else wb.active

    headers = [cell.value for cell in ws[1]]
    col_idx = {name: idx for idx, name in enumerate(headers)}

    required = ["EMPNO", "SERVICE_LINE", "ONSITE_OFFSHORE", "DERIVED_EMP_CITY", "SR_MANDATORY_SKILL"]
    for col in required:
        if col not in col_idx:
            raise ValueError(f"Column '{col}' not found in BenchData headers: {headers}")

    records = []
    for row in ws.iter_rows(min_row=2, values_only=True):
        if row[col_idx["EMPNO"]] is None:
            continue
        records.append({
            "EMPNO": row[col_idx["EMPNO"]],
            "SERVICE_LINE": str(row[col_idx["SERVICE_LINE"]] or "").strip().upper(),
            "ONSITE_OFFSHORE": str(row[col_idx["ONSITE_OFFSHORE"]] or "").strip().upper(),
            "CITY": str(row[col_idx["DERIVED_EMP_CITY"]] or "").strip().upper(),
            "SKILL": str(row[col_idx["SR_MANDATORY_SKILL"]] or "").strip(),
        })
    return records


def get_sl_from_filename(filename):
    match = re.search(r"(SL\d+)", filename, re.IGNORECASE)
    if not match:
        raise ValueError(f"Could not detect SL number from filename: {filename}")
    return match.group(1).upper()


def process_offshore_file(filepath, bench_records):
    sl_value = get_sl_from_filename(os.path.basename(filepath))

    # Filter bench records for this service line + offshore (plain loop, no pandas)
    subset = [
        r for r in bench_records
        if r["SERVICE_LINE"] == sl_value and r["ONSITE_OFFSHORE"] == "OFFSHORE"
    ]

    wb = load_workbook(filepath)
    ws = wb[SHEET_NAME] if SHEET_NAME in wb.sheetnames else wb.active

    header_row = 1
    existing_headers = {cell.value: cell.column for cell in ws[header_row] if cell.value}

    if COUNT_HEADER not in existing_headers:
        count_col = ws.max_column + 1
        ws.cell(row=header_row, column=count_col, value=COUNT_HEADER)
    else:
        count_col = existing_headers[COUNT_HEADER]

    if EMP_HEADER not in existing_headers:
        emp_col = ws.max_column + 1
        ws.cell(row=header_row, column=emp_col, value=EMP_HEADER)
    else:
        emp_col = existing_headers[EMP_HEADER]

    last_skill = None
    for row in range(header_row + 1, ws.max_row + 1):
        skill_cell = ws.cell(row=row, column=1).value
        city_cell = ws.cell(row=row, column=2).value

        if skill_cell is not None and str(skill_cell).strip() != "":
            last_skill = str(skill_cell).strip()

        if city_cell is None or str(city_cell).strip() == "":
            continue

        skill = last_skill
        city = str(city_cell).strip().upper()

        matched = [r for r in subset if r["SKILL"] == skill and r["CITY"] == city]

        count = len(matched)
        emp_numbers = ",".join(str(r["EMPNO"]) for r in matched)

        ws.cell(row=row, column=count_col, value=count)
        ws.cell(row=row, column=emp_col, value=emp_numbers if count > 0 else "")

    wb.save(filepath)
    print(f"[OFFSHORE] {os.path.basename(filepath)} -> SL={sl_value}, bench rows available: {len(subset)}")

## Run

In [ ]:
bench_records = load_bench_data(BENCH_DATA_PATH)
files = glob.glob(os.path.join(OFFSHORE_DIR, "SL*_OFFSHORE_CITY_Final_Predictions.xlsx"))

if not files:
    print("No offshore files found. Check OFFSHORE_DIR path.")
else:
    for f in sorted(files):
        process_offshore_file(f, bench_records)
    print("Offshore processing complete.")